In [1]:
# %load 2MedSAM_inpu_v1.1.py
"""
build_h5_dataset.py

将 dataset/ 下按类别/设备/病人的 MRI (ADC/DWI/T2 .nii/.nii.gz) 预处理并保存为单个 HDF5 文件。

标签映射（在此脚本中固定为）：
0: benign_prostate_cancer
1: non_significant_prostate_cancer
2: normal_prostate
3: significant_prostate_cancer

输出 HDF5 结构（每个病人一个 group，name 为 patient_index）：
    /<patient_index>/
        label   : int scalar (0..3)
        ADC     : (16,224,224,1) float32 (values normalized in [0,1])
        DWI     : (16,224,224,1) float32
        T2      : (16,224,224,1) float32
        attrs: orig_shape_ADC, orig_shape_DWI, orig_shape_T2 (strings)

脚本会打印每个读取到的原始体积形状，示例输出：
    Processing patient index 0: /path/to/.../patient123
    ADC original shape (Z, H, W): (30, 384, 384)
    DWI original shape (Z, H, W): (28, 384, 384)
    T2 original shape (Z, H, W):  (40, 448, 448)

若某病人缺少任一模态文件，脚本会跳过该病人并打印警告。
"""

import os
import sys
import numpy as np
import SimpleITK as sitk
from skimage import transform
import h5py
from tqdm import tqdm
import glob
from concurrent.futures import ProcessPoolExecutor, as_completed
import psutil
import gc
# ---------- 配置区 ----------
DATA_ROOT = "/data/users/lly/projects/PCa-HSD-LSDT/dataset/v1.0"               # 根目录：按你给出的结构放置
OUTPUT_H5 = "patients_dataset_v1.0.h5"   # 输出 h5 文件
MODALITIES = ["DWI", "ADC", "T2"]   # 需要的模态文件名（不含路径），脚本会在 patient folder 下查找 <MODALITY>.nii 或 .nii.gz
MID_SLICES = 16                     # 每个模态取的中间切片数
TARGET_SIZE = (224, 224)            # 重采样目标 HxW（仅 224 版本，无 1024）
BATCH_SIZE = 20                     # 每次处理的病人数，控制内存使用
# --------------------------------

# label -> 类别名 的映射（固定）
LABEL_MAP = {
    "benign_prostate_cancer": 1,
    "non_significant_prostate_cancer": 2,
    "normal_prostate": 0,
    "significant_prostate_cancer": 3,
}

# 允许的 nii 扩展名
NIIFMT = [".nii", ".nii.gz"]

def find_modality_file(patient_dir, modality):
    """
    在 patient_dir 下寻找 modality 名称的 nii/nii.gz 文件（大小写不敏感）。
    优先匹配 modality + ext（例如 ADC.nii.gz / ADC.nii），也接受 modality 小写/大写。
    返回第一个匹配的完整路径或 None。
    """
    # try exact
    for ext in NIIFMT:
        p = os.path.join(patient_dir, modality + ext)
        if os.path.exists(p):
            return p
    # case-insensitive search
    files = glob.glob(os.path.join(patient_dir, "*"))
    for f in files:
        fname = os.path.basename(f).lower()
        if fname.startswith(modality.lower()) and (fname.endswith(".nii") or fname.endswith(".nii.gz")):
            return f
    return None

def read_vol(path):
    """ 使用 SimpleITK 读取 nii 文件并返回 numpy 体积 (Z,H,W) """
    img = sitk.ReadImage(path)
    arr = sitk.GetArrayFromImage(img)  # SimpleITK gives (Z,H,W)
    return arr, img.GetSpacing()

def take_center_slices(vol, num_slices):
    """
    从 vol (Z,H,W) 中取中间 num_slices 张切片（沿 Z 轴）。
    若 Z < num_slices，则以边缘切片重复补齐。
    返回 shape (num_slices, H, W) 的数组。
    """
    Z = vol.shape[0]
    if Z >= num_slices:
        start = (Z - num_slices) // 2
        return vol[start:start + num_slices]
    else:
        # pad by repeating last slice
        pad_needed = num_slices - Z
        pads = np.repeat(vol[-1][None, ...], pad_needed, axis=0)
        return np.concatenate([vol, pads], axis=0)

def preprocess_slice_gray_to_1ch(slice2d, target_size):
    """
    对单张灰度切片做：
      - 复制为 1 通道
      - 重采样到 target_size (H, W)（使用 cubic 插值）
      - per-slice min/max 归一化到 [0,1] (float32)
    返回 shape (H, W, 1) dtype float32
    """
    # 保证为 float
    sl = slice2d.astype(np.float32)
    # 复制 1 通道
    sl1 = np.repeat(sl[:, :, None], 1, axis=-1)  # (H, W, 1)
    # resize 到 (H,W,1)
    out_shape = (target_size[0], target_size[1], 1)
    resized = transform.resize(
        sl1,
        out_shape,
        order=3,
        preserve_range=True,
        anti_aliasing=True,
        mode="constant",
    )
    # per-slice min-max normalize
    mn = resized.min()
    mx = resized.max()
    if mx - mn > 1e-8:
        norm = (resized - mn) / (mx - mn)
    else:
        norm = np.zeros_like(resized, dtype=np.float32)
    return norm.astype(np.float32)

def process_patient(patient_dir):
    """
    处理单个病人目录：
      - 读取 ADC/DWI/T2；若任一缺失则返回 None
      - 打印各模态原始体积形状 (Z,H,W)
      - 提取中间 MID_SLICES 张切片并对每张做 preprocess_slice_gray_to_1ch
      - 返回 dict: { 'ADC': arr, 'DWI': arr, 'T2': arr, 'orig_shapes': { ... } }
      where arr shape == (MID_SLICES, TARGET_H, TARGET_W, 1), dtype float32
    """
    vols = {}
    orig_shapes = {}
    spacings = {}
     # 提取机器名
    machine = os.path.basename(os.path.dirname(patient_dir))
    for mod in MODALITIES:
        fpath = find_modality_file(patient_dir, mod)
        if fpath is None:
            print(f"[WARN] patient {patient_dir} missing modality {mod}, skip this patient.")
            return None
        vol, spacing = read_vol(fpath)
        # 打印原始形状
        # print(f"    {mod} original shape (Z, H, W): {vol.shape}")
        orig_shapes[mod] = vol.shape
        spacings[mod] = spacing
        # take center slices
        vol16 = take_center_slices(vol, MID_SLICES)  # (MID_SLICES, H, W)
        # process each slice to 224x224 only
        processed_slices = []
        for i in range(vol16.shape[0]):
            sl = vol16[i]
            sl_224 = preprocess_slice_gray_to_1ch(sl, TARGET_SIZE)
            processed_slices.append(sl_224)
        vols[mod] = np.stack(processed_slices, axis=0)
    return {
        "vols": vols,
        "orig_shapes": orig_shapes,
        "spacings": spacings,
        "machine": machine,
    }

def gather_patients(root_dir):
    """
    遍历 root_dir，按 LABEL_MAP 的键顺序收集每个类别下的病人目录。
    返回 list of tuples: (patient_dir, label_name)
    """
    entries = []
    for label_name in LABEL_MAP.keys():
        class_dir = os.path.join(root_dir, label_name)
        if not os.path.isdir(class_dir):
            print(f"[WARN] class directory not found: {class_dir}, skipping.")
            continue
        print(class_dir)
        # 深度遍历 class_dir: 设备子目录下的 patient folders，或 class 直接包含 patient folders
        # 我们列举 class_dir 下所有子目录，递归找到包含模态 nii 的最底层目录
        for root, dirs, files in os.walk(class_dir):
            # heuristic: if any nii file present in this folder, treat as patient folder
            has_nii = any(fname.lower().endswith(".nii") or fname.lower().endswith(".nii.gz") for fname in files)
            if has_nii:
                entries.append((root, label_name))
    return entries

def memory_usage():
    """返回当前进程的内存使用量(MB)"""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss/ 1024 / 1024 

def main():
    patients = gather_patients(DATA_ROOT)
    print(f"Found {len(patients)} candidate patient folders")
    print(f"初始内存: {memory_usage():.1f}MB")
    
    # 分批处理
    total_batches = (len(patients) + BATCH_SIZE - 1) // BATCH_SIZE
    global_patient_index = 0
    
    # 以追加模式创建HDF5文件
    with h5py.File(OUTPUT_H5, "w") as f:
        for batch_idx in range(total_batches):
            batch_start = batch_idx * BATCH_SIZE
            batch_end = min((batch_idx + 1) * BATCH_SIZE, len(patients))
            batch_patients = patients[batch_start:batch_end]
            
            print(f"\n处理批次 {batch_idx + 1}/{total_batches} (病人 {batch_start}-{batch_end-1})")
            print(f"批次开始内存: {memory_usage():.1f}MB")
            
            # 处理当前批次
            batch_processed = []
            for pdir, lbl in tqdm(batch_patients, desc=f"批次 {batch_idx+1}"):
                try:
                    res = process_patient(pdir)
                    if res is not None:
                        batch_processed.append((pdir, lbl, res))
                except Exception as e:
                    print(f"[ERROR] processing {pdir} failed: {e}")
            
            # 保存当前批次到HDF5
            for pdir, lbl, res in batch_processed:
                grp = f.create_group(str(global_patient_index))
                label_int = LABEL_MAP[lbl]
                grp.create_dataset("label", data=np.int64(label_int))
                grp.attrs["machine_name"] = res["machine"]
                
                for mod in MODALITIES:
                    data = res["vols"][mod]  # 仅 224x224 数据
                    grp.create_dataset(mod, data=data, compression="gzip", compression_opts=4)
                    grp.attrs[f"orig_shape_{mod}"] = str(res["orig_shapes"][mod])
                    grp.attrs[f"spacing_{mod}"] = str(res["spacings"][mod])
                
                grp.attrs["patient_dir"] = pdir
                global_patient_index += 1
            
            # 清理当前批次内存
            del batch_processed
            gc.collect()
            
            print(f"批次完成，已处理 {global_patient_index} 个病人")
            print(f"批次结束内存: {memory_usage():.1f}MB")
    
    print(f"\n完成! 总共保存 {global_patient_index} 个病人到 {OUTPUT_H5}")

if __name__ == "__main__":
    main()


/data/users/lly/projects/PCa-HSD-LSDT/dataset/v1.0/benign_prostate_cancer
/data/users/lly/projects/PCa-HSD-LSDT/dataset/v1.0/non_significant_prostate_cancer
/data/users/lly/projects/PCa-HSD-LSDT/dataset/v1.0/normal_prostate
/data/users/lly/projects/PCa-HSD-LSDT/dataset/v1.0/significant_prostate_cancer
Found 344 candidate patient folders
初始内存: 178.3MB

处理批次 1/18 (病人 0-19)
批次开始内存: 179.2MB


批次 1: 100%|██████████| 20/20 [02:16<00:00,  6.83s/it]


批次完成，已处理 20 个病人
批次结束内存: 410.4MB

处理批次 2/18 (病人 20-39)
批次开始内存: 410.4MB


批次 2: 100%|██████████| 20/20 [01:40<00:00,  5.01s/it]


批次完成，已处理 40 个病人
批次结束内存: 431.7MB

处理批次 3/18 (病人 40-59)
批次开始内存: 431.7MB


...

批次完成，已处理 320 个病人
批次结束内存: 483.8MB

处理批次 17/18 (病人 320-339)
批次开始内存: 483.8MB


批次 17: 100%|██████████| 20/20 [01:35<00:00,  4.76s/it]


批次完成，已处理 340 个病人
批次结束内存: 486.7MB

处理批次 18/18 (病人 340-343)
批次开始内存: 486.7MB


批次 18: 100%|██████████| 4/4 [00:07<00:00,  1.92s/it]


批次完成，已处理 344 个病人
批次结束内存: 410.6MB

完成! 总共保存 344 个病人到 patients_dataset_v1.0.h5
